# 第3回：推論統計基礎 〜 科学的根拠を証明する論理 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session03_inferential_statistics.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

0. **【はじめに】なぜ「推論統計」が必要なのか？ (15分)**
   - 目に見えるデータから、目に見えない「真実」を探る旅。
1. **【講義1】サンプリング理論と中心極限定理 (10分)**
   - 標本から母集団を推測するための土台。
2. **【講義2】仮説検定の論理：帰無仮説の裁判 (15分)**
   - 「差がない」と仮定して矛盾を突く、背理法の論理。
3. **【講義3】検定手法の選択：パラメトリック vs ノンパラメトリック (10分)**
   - 正規分布の有無が分ける、解析の「運命の分かれ道」。
4. **【実習1】独立 2 群の平均値比較 (15分)**
   - t 検定とマン・ホイットニーの U 検定。
5. **【実習2】対応のある比較と多群比較 (15分)**
   - 前後比較と ANOVA（分散分析）。
6. **【実習3】比率の比較と効果量 (10分)**
   - カイ二乗検定と、p 値に頼らない「効果」の大きさ。

---

## 0. 【導入】なぜ「推論統計」が必要なのか？

### 0.1 目に見えるのは「氷山の一角」に過ぎない
私たちは日々、多くのデータに囲まれています。「この薬を飲んだ 50 人のうち 40 人が治った」「新しく開発した AI の正解率が 85% だった」。これらはすべて、**実際に観測された事実**です。

しかし、科学者やデータサイエンティストが本当に知りたいのは、「目の前の 50 人がどうだったか」ではありません。本当に知りたいのは、**「これからこの薬を飲む世界中の何万人もの患者さんに、本当に効果があるのか？」**ということです。

- 私たちが手元に持っているデータ：**標本 (Sample)**
- 私たちが本当に知りたい対象の全体：**母集団 (Population)**

世界中のすべての人を調査すること（全数調査）は物理的・経済的に不可能です。そのため、私たちは「一部のデータ（標本）」から「全体（母集団）」の姿を**推論 (Inference)** する必要があります。これが「推論統計」という学問の出発点です。

### 0.2 「偶然」というノイズとの戦い
推論を難しくする最大の敵は、**偶然（タマタマ起きたこと）**です。

例えば、ある新しい治療法を試した 10 人が全員回復したとしましょう。これは一見素晴らしい結果ですが、実は「たまたま回復力の強い 10 人が集まっただけ」あるいは「何もしなくても治る時期だっただけ」かもしれません。コインを 10 回投げて、たまたま 10 回連続で表が出る確率が 0 ではないのと同様です。

データサイエンスにおける推論統計の役割は、**「この結果は、本当に意味のある（実力による）結果なのか、それとも単なるラッキー（偶然の誤差）なのか？」を、確率という物差しで客観的に測ること**にあります。

### 0.3 「証拠」の確からしさを数値化する
「なんとなく効果がありそうだ」という直感や経験は、しばしば私たちを誤った判断へと導きます。推論統計という強力な武器を備えることで、私たちは以下のような主張を科学的に行うことができます。

1. **仮説検定**: 「この差が偶然で起きる確率は 5% 以下である。したがって、この治療法には（偶然では片付けられない）有意な効果があると判断する」
2. **推定**: 「世界中の患者さんに使った場合、生存率は 92% 前後に収まり、そのブレ幅は 90% 〜 94% の間だろう（95% 信頼区間）」

数式を使わずに本質を言えば、推論統計とは**「不確かな世界において、どれくらい自信を持って結論を言って良いかを測るための論理」**です。この論理があるからこそ、私たちは限られた実験データから、新しい薬を承認したり、新しい技術を社会に導入したりできるのです。

---

## 1. セットアップ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style='whitegrid', context='talk', palette='muted')

print("統計解析エンジン、フルパワーで起動。")

## 2. 【講義】検定手法の選択：パラメトリック vs ノンパラメトリック

### 2.1 運命の分かれ道：正規性の確認
データが「綺麗なベル型の分布（正規分布）」をしていると仮定できる場合の手法を **パラメトリック検定** 、仮定できない場合の手法を **ノンパラメトリック検定** と呼びます。医学データ（特に検査値やアンケートスコア）は非正規分布であることが多いため、この使い分けは極めて重要です。

| 比較対象 | パラメトリック (正規分布あり) | ノンパラメトリック (正規分布なし) |
| :--- | :--- | :--- |
| **注目する指標** | 平均値 (Mean) | 中央値 (Median) / 順位 (Rank) |
| **独立 2 群の比較** | **Welch の t 検定** | **Mann-Whitney の U 検定** |
| **対応のある比較** | **対応のある t 検定** | **Wilcoxon の符号付順位検定** |
| **3 群以上の比較** | **ANOVA (分散分析)** | **Kruskal-Wallis 検定** |

### 2.2 正規性の判断 (Shapiro-Wilk 検定)
目視（ヒストグラム）だけでなく、数学的に正規性をテストします。
- **帰無仮説 $H_0$**: データは正規分布に従っている。
- **$p < 0.05$ の時**: 正規分布から有意に外れている $ightarrow$ **ノンパラへ！**

---

### 【実践】正規性のチェックと検定の選択

In [ ]:
# 擬似データ：非正規分布（右に裾が長い、所得や一部の検査値のような分布）
np.random.seed(42)
group_non_normal = np.random.lognormal(mean=0, sigma=1, size=50)
group_control = np.random.lognormal(mean=0.5, sigma=1, size=50)

# 1. Shapiro-Wilk 検定で正規性を確認
_, p_norm = stats.shapiro(group_non_normal)
print(f"Shapiro-Wilk p-value: {p_norm:.5f}")

if p_norm < 0.05:
    print("【判定】正規分布ではない。ノンパラメトリック検定を選択すべき。")
    # 2. Mann-Whitney の U 検定を実行
    u_stat, p_u = stats.mannwhitneyu(group_non_normal, group_control)
    print(f"Mann-Whitney U-test p-value: {p_u:.5f}")
else:
    print("【判定】正規分布とみなせる。パラメトリック（t検定）を選択。")
    t_stat, p_t = stats.ttest_ind(group_non_normal, group_control, equal_var=False)
    print(f"Welch t-test p-value: {p_t:.5f}")

## 3. 【実習1】独立 2 群の比較（t 検定 vs U 検定）

医学論文では、データの分布に応じてこれらが見事に使い分けられます。中央値を比較しているのか、平均値を比較しているのかを常に意識しましょう。

## 4. 【実習2】対応のある比較と多群比較

同一人物の治療前後の比較（対応あり）や、3 群以上のグループ間の差（ANOVA）を扱います。

In [ ]:
# 前後のデータ
before = [10, 12, 11, 15, 10, 9, 8, 12, 11, 13] 
after  = [12, 14, 12, 18, 13, 11, 10, 15, 12, 16]

# 正規分布に従わない可能性がある場合（小標本など）
res_wilcoxon = stats.wilcoxon(before, after)
print(f"Wilcoxon Signed-Rank Test p-value: {res_wilcoxon.pvalue:.5f}")

## 5. 【実習3】カイ二乗検定と効果量

比率を比べるカイ二乗検定と、p 値に頼りすぎないための指標「効果量（Cohen's d など）」を学びます。

---

## ✏️ 本日の最終演習問題 (The Choice Challenge)

**シナリオ**: 患者 30 名のリハビリ前後の歩行距離データがあります。

### 課題 1: 手法の選定ルールを記述せよ
「独立した 2 つのグループ（若年・高齢）の平均血圧を比較したい」という時、どのようなステップで検定手法を決定すべきか、3 つのステップで説明してください。

### 課題 2: 実践解析
提供されたデータ `walk_dist` に対して Shapiro-Wilk 検定を行い、その結果に基づいて t 検定か U 検定のいずれか適切な方を実行してください。

### 課題 3: 理論の整理
「サンプルサイズが非常に大きい（例：1万人）」場合、正規性の検定をすることにどのような意味があるか（またはないか）、中心極限定理の観点から考察してください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)